# EPInformer — K562 end-to-end walkthrough

This notebook is a **guided runbook** for reproducing EPInformer on **K562**, from raw ENCODE data
to the final expression metrics. It trains **two models, in order**:

1. **Enhancer-activity encoder** — predicts 256 bp enhancer activity (H3K27ac·DNase) from sequence.
2. **Gene-expression model** (`EPInformer_v2`) — predicts RNA / CAGE from a gene's promoter plus its
   ABC-nominated enhancers, reusing the **frozen** encoder as the sequence backbone.

> **Do Part 1 (encoder) first, then Part 2 (expression)** — the expression model loads the encoder.

## What you should get (K562, 12-fold leave-chromosome-out, pooled Pearson R)

| Stage | Metric | Target |
|---|---|---|
| Encoder, single-reverse evaluation | Pearson R | **~0.734** |
| Encoder, forward + RC averaged evaluation | Pearson R | **~0.740** |
| Expression — RNA | Pearson R | **~0.86** |
| Expression — CAGE | Pearson R | **~0.87** |

## How to use this notebook

- **Everything runs on the HPC** (`luminary`, `/lustre/grp/zyjlab/linjc/epinformer_reproduce`) inside
  the **`EPInformer_env`** conda env. Full training requires a CUDA GPU; the training cells stop
  instead of silently starting a long CPU/MPS run.
- The **training cells (1b, 2b) run one fold in-process** so you can inspect the actual model and
  training loop. Use the displayed SLURM array commands for the complete 12-fold run.
- **Preprocessing cells are safe by default:** they preview commands and only download or submit
  jobs after you explicitly change their opt-in flags. CPU/memory-heavy work runs through SLURM.
- Evaluation cells refuse to report a headline pooled metric until all 12 fold files are present.
- The final inspection cells can also run locally after the small result files have been synced.
- The first code cell sets `HDF5_USE_FILE_LOCKING=FALSE` in the notebook kernel before HDF5-backed
  modules are imported.

## 0 · Prerequisites & one-time setup

Start Jupyter **from the repository root on the HPC** with the EPInformer environment:

```bash
cd /lustre/grp/zyjlab/linjc/epinformer_reproduce
conda activate EPInformer_env
python -m ipykernel install --user --name ep_env --display-name 'EPInformer (ep_env)'
jupyter lab
```

Select the **EPInformer (ep_env)** kernel for this notebook. Activating conda inside a `%%bash` cell
would affect only that temporary subprocess, not the Python kernel, so environment activation is
intentionally kept outside the notebook. The local `environment.yml` names the same environment
`epinformer_repro`.

**Reference & label files** (place once):
- `hg38.fa` → `data/reference/hg38/hg38.fa` *(not downloaded by any script)*
- ABC reference: `bash scripts/download_abc_reference.sh data/reference/hg38`
- Expression labels + Xpresso features + the 12-fold CV split: download Zenodo **13232430**, then
  unzip `expression_data.zip` into `data/`.

The next cell checks the kernel and required paths without downloading or training anything.

In [ ]:
# --- notebook-kernel preflight: run before importing torch/h5py-backed modules ---
import os
import sys
from pathlib import Path

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
ROOT = Path.cwd().resolve()
if not (ROOT / "EPInformer" / "models.py").is_file():
    raise RuntimeError("Start Jupyter from the epinformer_reproduce repository root.")
try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "This kernel is missing PyTorch. Select the EPInformer (ep_env) kernel."
    ) from exc

required = {
    "hg38 FASTA": ROOT / "data/reference/hg38/hg38.fa",
    "Xpresso gene bounds": ROOT / "data/reference/hg38/XpressoGeneBounds.hg38.bed",
    "chromosome sizes": ROOT / "data/reference/hg38/GRCh38_EBV.chrom.sizes.tsv",
    "K562 quantile reference": ROOT / "data/reference/hg38/EnhancersQNormRef.K562.txt",
    "base expression/features table": ROOT / "data/GM12878_K562_18377_gene_expr_fromXpresso.csv",
    "strand-aware expression table": ROOT / "data/GM12878_K562_18377_gene_expr_fromXpresso_with_sequence_strand.csv",
    "CV split": ROOT / "data/leave_chrom_out_crossvalidation_split_18377genes.csv",
}
missing = [f"{label}: {path}" for label, path in required.items() if not path.is_file()]
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Repository:", ROOT)
print("HDF5_USE_FILE_LOCKING:", os.environ["HDF5_USE_FILE_LOCKING"])
if missing:
    raise FileNotFoundError("Setup still required:\n  - " + "\n  - ".join(missing))
print("All configured reference and label files: OK")

---
# Part 1 · Enhancer-activity encoder  *(do this first)*

Predicts 256 bp enhancer activity from sequence. Target = **`log2(0.1 + Activity)`** where
**`Activity = sqrt(H3K27ac_RPM · DNase_RPM)`**, using **5 bins** at summit ±192·{−2..2}, **L1KL**
loss, batch 256, 12-fold leave-chromosome-out.

K562 deliberately uses one H3K27ac replicate: the three available replicates differ greatly in
sequencing depth, and equal-weight pooling reduces reproduction accuracy.

### 1a · Data preprocessing — ENCODE → ABC → 256 bp activity CSV

The next cell can download the pinned K562 inputs into `data/`, preview the ABC command, and submit
the CPU/memory-heavy links stage through SLURM. Both downloads and submission require explicit
opt-in, so **Run All will not start a large transfer or queue a 128 GB job**.

**ENCODE accessions (K562, Roadmap E123):** DNase `ENCFF257HEE` · H3K27ac `ENCFF232RQF` ·
narrowPeak `ENCFF544LXB` · Hi-C `ENCFF621AIY`.

In [ ]:
%%bash
set -euo pipefail
export HDF5_USE_FILE_LOCKING=FALSE

# Safety switches: change deliberately, then rerun this cell.
RUN_DOWNLOADS=0
SUBMIT_LINKS_JOB=0
K562_PEAK=reference/K562_H3K27ac.ENCFF544LXB.narrowPeak

if [[ "$RUN_DOWNLOADS" == 1 ]]; then
  # Use the pinned reproduction manifest; do not rely on a changing API query.
  python scripts/download_encode_data.py --from-manifest config/k562_reproduction_inputs.json
  if [[ ! -f "$K562_PEAK" ]]; then
    python scripts/download_encode_data.py \
      --from-manifest config/encoder_narrowpeaks.json --cell-types K562
    gunzip -f "${K562_PEAK}.gz"
  fi
else
  echo "Downloads skipped (set RUN_DOWNLOADS=1 after confirming storage and transfer policy)."
  echo "  python scripts/download_encode_data.py --from-manifest config/k562_reproduction_inputs.json"
fi

# Fast configuration preview only; the real links stage belongs on cpu1.
python run_pipeline.py --config config/config.yaml --samples K562 --stages links --dry-run
K562_INPUTS=(
  data/K562/DNase/ENCFF257HEE.bam
  data/K562/H3K27ac/ENCFF232RQF.bam
  data/K562/HiC/ENCFF621AIY.hic
  "$K562_PEAK"
)
if [[ "$SUBMIT_LINKS_JOB" == 1 ]]; then
  for input in "${K562_INPUTS[@]}"; do
    [[ -f "$input" ]] || { echo "Missing required K562 input: $input" >&2; exit 1; }
  done
  mkdir -p log_cpu
  SAMPLES=K562 STAGES=links sbatch slurm/run_pipeline_cpu.slurm
else
  echo "Links job not submitted (set SUBMIT_LINKS_JOB=1 when inputs are ready)."
  echo "  SAMPLES=K562 STAGES=links sbatch slurm/run_pipeline_cpu.slurm"
fi
# Outputs after the job completes:
#   batch_output/K562/links/K562_peak_5bins_around_summit_activity_sequence.csv
#   batch_output/K562/links/{EnhancerList,GeneList}.txt + Predictions/

### 1b · Train the encoder — in-cell (5-bin / L1KL recipe, one fold)

The cell below runs the **actual training loop** in-process for the selected `FOLD`. It uses the
reproduction seed, requires CUDA by default, writes the best checkpoint to
`results/seqencoder/K562_repro/checkpoints/`, and saves a fold-qualified prediction file.

For the complete result, run all 12 folds in parallel rather than repeatedly editing this cell:

```bash
CELL=K562 OUTPUT_DIR=results/seqencoder/K562_repro \
  sbatch slurm/train_seqencoder_12fold.slurm
```

Set `ALLOW_NON_CUDA_SMOKE_TEST = True` only for a deliberately short local test, together with a
small `EPOCHS` value. A CPU/MPS smoke test is not expected to reproduce the headline metric.

In [ ]:
# --- Part 1b: train the enhancer-activity encoder (one fold, in-process) ---
import os
import random
import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr

from EPInformer.models import enhancer_predictor_256bp
from train_seqEncoder import PeakActivityDataset, train, test

CELL     = "K562"
FOLD     = 1                       # choose one fold here; use SLURM for all 12
EPOCHS   = 50                      # use 1-5 only for an explicit smoke test
SEED     = 66
ALLOW_NON_CUDA_SMOKE_TEST = False
DATA_CSV = "batch_output/K562/links/K562_peak_5bins_around_summit_activity_sequence.csv"
ENC_OUT  = "results/seqencoder/K562_repro"

if FOLD not in range(1, 13):
    raise ValueError("FOLD must be between 1 and 12")
device = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)
if device != "cuda" and not ALLOW_NON_CUDA_SMOKE_TEST:
    raise RuntimeError(
        f"Full training requires CUDA; detected {device}. "
        "Use the HPC GPU kernel, or explicitly enable a short non-CUDA smoke test."
    )
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"device: {device}  seed: {SEED}  fold: {FOLD}")

# leave-chromosome-out fold assignment (the split file the scripts use)
split = pd.read_csv("data/leave_chrom_out_crossvalidation_split_18377genes.csv", index_col=0)
split["chrom"] = "chr" + split["chrom"].astype(str)
fk = f"fold_{FOLD}"
train_chrom = list(split[split[fk] == "train"]["chrom"].unique())
valid_chrom = list(split[split[fk] == "valid"]["chrom"].unique())
test_chrom  = list(split[split[fk] == "test"]["chrom"].unique())

full_df  = pd.read_csv(DATA_CSV)
train_ds = PeakActivityDataset(CELL, train_chrom, strand="both",    dataframe=full_df)
valid_ds = PeakActivityDataset(CELL, valid_chrom, strand="forward", dataframe=full_df)
test_ds  = PeakActivityDataset(CELL, test_chrom,  strand="reverse", dataframe=full_df)

model = enhancer_predictor_256bp().to(device)
model.name = f"enhancer_predictor_H3K27ac_256bp_{CELL}"

# This cell uses the documented single-reverse protocol (~0.734 pooled over 12 folds).
train(model, train_ds, valid_dataset=valid_ds, fold_i=FOLD,
      saved_model_path=f"{ENC_OUT}/checkpoints", model_name=model.name,
      learning_rate=5e-4, batch_size=256, EPOCHS=EPOCHS, device=device, loss_type="l1kl")

preds = test(model, test_ds, fold_i=FOLD, model_name=model.name,
             saved_model_path=f"{ENC_OUT}/checkpoints", batch_size=128,
             device=device, rc_average=False)
os.makedirs(f"{ENC_OUT}/predictions", exist_ok=True)
preds.to_csv(f"{ENC_OUT}/predictions/"
             f"fold_{FOLD}_enhancer_predictor_H3K27ac_256bp_{CELL}_reverse_predictions.csv",
             index=False)
print(f"\nfold {FOLD} encoder Pearson R = {pearsonr(preds['preds'], preds['actual'])[0]:.4f}")

### 1c · Evaluate the encoder  *(single-reverse ≈ 0.734; fwd+RC ≈ 0.740)*

The cell checks for one prediction file per fold before running pooled evaluation. If you trained
only the walkthrough fold, it reports the missing folds instead of presenting a partial metric as
the 12-fold result. The default training predictions are **single-reverse**. For the headline
forward+RC protocol, re-score the same checkpoints into a separate directory:

```bash
CELL=K562 \
DATA_CSV=./batch_output/K562/links/K562_peak_5bins_around_summit_activity_sequence.csv \
CKPT_DIR=./results/seqencoder/K562_repro/checkpoints \
OUTPUT_DIR=./results/seqencoder/K562_repro_fwdRC \
  sbatch slurm/eval_seqencoder_fwdrc.slurm
```

Never mix the two protocols' prediction files in one directory.

In [ ]:
from pathlib import Path
import subprocess
import sys

encoder_pred_dir = Path(ENC_OUT) / "predictions"
missing_folds = [
    fold for fold in range(1, 13)
    if not list(encoder_pred_dir.glob(f"fold_{fold}_*_predictions.csv"))
]
if missing_folds:
    print(f"Encoder pooled evaluation skipped; missing folds: {missing_folds}")
    print("Run the 12-fold SLURM array, then rerun this cell.")
else:
    subprocess.run(
        [sys.executable, "evaluate.py", "encoder", "--pred_dir", ENC_OUT],
        check=True,
    )

---
# Part 2 · Gene-expression model (`EPInformer_v2`)  *(do this second)*

Reuses the Part 1 encoder (**frozen**) + the ABC enhancer–gene links from step 1a. The shipped config
is **`f3`**: `n_enh_feats = 3` (distance + activity + Hi-C) **+ promoter activity** (`use_prm_signal`).

### 2a · Build the gene HDF5 (ABC links → factored per-gene tensors)

Output → `batch_output/K562/encoding/K562_samples.h5` (promoter + enhancer one-hot sequence, and the
activity / DHS / distance / contact features). The next cell previews the command and submits the
CPU/memory-heavy encoding stage only after explicit opt-in.

In [ ]:
%%bash
set -euo pipefail
export HDF5_USE_FILE_LOCKING=FALSE
SUBMIT_ENCODING_JOB=0

python run_pipeline.py --config config/config.yaml --samples K562 --stages encoding --dry-run
if [[ "$SUBMIT_ENCODING_JOB" == 1 ]]; then
  mkdir -p log_cpu
  SAMPLES=K562 STAGES=encoding sbatch slurm/run_pipeline_cpu.slurm
else
  echo "Encoding job not submitted (set SUBMIT_ENCODING_JOB=1 after the links job succeeds)."
  echo "  SAMPLES=K562 STAGES=encoding sbatch slurm/run_pipeline_cpu.slurm"
fi
# Output after the job completes: batch_output/K562/encoding/K562_samples.h5

### 2b · Train `EPInformer_v2` (f3) — in-cell, one fold

The cell mirrors the maintained trainer: it loads promoter activity, creates the factored dataset,
applies the leave-chromosome-out split, loads the matching frozen encoder fold, and evaluates the
held-out genes. `EXPR_TYPE` may be `RNA` or `CAGE`; output paths are derived from it so the two tasks
cannot be mixed accidentally. CUDA is required unless an explicit short smoke test is enabled.

For the complete 12-fold run:

```bash
CELL=K562 EXPR_TYPE=RNA \
  PRETRAINED_DIR=results/seqencoder/K562_repro/checkpoints \
  OUTPUT_DIR=EPInformer_models/K562_repro_RNA_prm \
  sbatch slurm/train_epinformer_12fold.slurm
```

In [ ]:
# --- Part 2b: train EPInformer_v2 (f3) on the frozen encoder (one fold, in-process) ---
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Subset

from EPInformer.models import EPInformer_v2, enhancer_predictor_256bp
from train_EPInformer import promoter_enhancer_dataset, train, test

CELL       = "K562"
FOLD       = 1                     # choose one fold here; use SLURM for all 12
EXPR_TYPE  = "RNA"                 # "RNA" or "CAGE"
N_ENH_FEATS    = 3                 # f3 = distance + activity + Hi-C
USE_PRM_SIGNAL = True              # f3 adds promoter activity
EPOCHS     = 50                    # use 1-5 only for an explicit smoke test
SEED       = 66
ALLOW_NON_CUDA_SMOKE_TEST = False
H5_PATH    = "batch_output/K562/encoding/K562_samples.h5"
EXPR_CSV   = "data/GM12878_K562_18377_gene_expr_fromXpresso_with_sequence_strand.csv"
SPLIT_CSV  = "data/leave_chrom_out_crossvalidation_split_18377genes.csv"
GENE_LIST  = "batch_output/K562/links/GeneList.txt"
ENC_CKPTS  = "results/seqencoder/K562_repro/checkpoints"
EXPR_OUT   = f"EPInformer_models/{CELL}_repro_{EXPR_TYPE}_prm"
MAX_N_ENH, DIST_THR, LR, BATCH = 60, 100_000, 1e-4, 50

if FOLD not in range(1, 13):
    raise ValueError("FOLD must be between 1 and 12")
if EXPR_TYPE not in {"RNA", "CAGE"}:
    raise ValueError("EXPR_TYPE must be 'RNA' or 'CAGE'")
device = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)
if device != "cuda" and not ALLOW_NON_CUDA_SMOKE_TEST:
    raise RuntimeError(
        f"Full training requires CUDA; detected {device}. "
        "Use the HPC GPU kernel, or explicitly enable a short non-CUDA smoke test."
    )
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"device: {device}  seed: {SEED}  fold: {FOLD}  target: {EXPR_TYPE}")
print("output:", EXPR_OUT)

# promoter activity = sqrt(DHS * H3K27ac) at TSS +/- 1kb  (the use_prm_signal feature)
prm_df = None
if USE_PRM_SIGNAL:
    prm_df = pd.read_csv(GENE_LIST, sep="\t", index_col="name")
    dhs = "DHS.RPKM.TSS1Kb" if "DHS.RPKM.TSS1Kb" in prm_df.columns else "DHS.RPM.TSS1Kb"
    h3k = "H3K27ac.RPKM.TSS1Kb" if "H3K27ac.RPKM.TSS1Kb" in prm_df.columns else "H3K27ac.RPM.TSS1Kb"
    prm_df["promoter_activity"] = np.sqrt(prm_df[dhs] * prm_df[h3k])

# auto-detect Xpresso gene-structure features (drives EPInformer_v2 useFeat)
rna_cols = ["UTR5LEN_log10zscore", "CDSLEN_log10zscore", "INTRONLEN_log10zscore",
            "UTR3LEN_log10zscore", "UTR5GC", "CDSGC", "UTR3GC", "ORFEXONDENSITY"]
use_rna_feats = any(c in pd.read_csv(EXPR_CSV, nrows=1).columns for c in rna_cols)

# factored dataset + leave-chromosome-out split -> train/valid/test Subsets
ds = promoter_enhancer_dataset(
        h5_path=H5_PATH, expr_csv=EXPR_CSV, cell_type=CELL, expr_type=EXPR_TYPE,
        n_enh_feats=N_ENH_FEATS, distance_thr=DIST_THR, max_n_enh=MAX_N_ENH,
        use_prm_signal=USE_PRM_SIGNAL, rm_prm_seq=False, promoter_activity_df=prm_df,
        strand_aware=False, rm_self_promoter=False)

split = pd.read_csv(SPLIT_CSV, index_col=0)
fk = f"fold_{FOLD}"
ensid = pd.DataFrame({"idx": np.arange(len(ds._ensid))}, index=ds._ensid)
def _idx(tag):
    keep = ensid.index.intersection(split[split[fk] == tag].index)
    return ensid.loc[keep]["idx"]
train_ds, valid_ds, test_ds = Subset(ds, _idx("train")), Subset(ds, _idx("valid")), Subset(ds, _idx("test"))
print(f"fold {FOLD}: train {len(train_ds)}  valid {len(valid_ds)}  test {len(test_ds)}")

# frozen pretrained encoder -> EPInformer_v2 (f3)
ckpt_path = f"{ENC_CKPTS}/fold_{FOLD}_best_enhancer_predictor_H3K27ac_256bp_{CELL}_checkpoint.pt"
enc = enhancer_predictor_256bp()
enc.load_state_dict(torch.load(ckpt_path, weights_only=False)["model_state_dict"])
model = EPInformer_v2(
        n_extraFeat=N_ENH_FEATS, pre_trained_encoder=enc.encoder, useFeat=use_rna_feats,
        out_dim=64, n_enhancer=MAX_N_ENH, useBN=False, usePromoterSignal=USE_PRM_SIGNAL,
        n_encoder=3, head=4, device=device).to(device)
for name, parameter in model.named_parameters():
    if name.startswith("seq_encoder"):
        parameter.requires_grad = False
rna_flag = "rnafeats" if use_rna_feats else "nornafeats"
prm_flag = "prmsig" if USE_PRM_SIGNAL else "nonprmsig"
model.name = (
    f"EPInformerV2.preTrainedConv.{CELL}.{EXPR_TYPE}.{MAX_N_ENH}enhs."
    f"{N_ENH_FEATS}feats.{rna_flag}.{prm_flag}.nonrmprmseq.100kb2TSS"
)

train(model, train_ds, valid_dataset=valid_ds, fold_i=FOLD, saved_model_path=EXPR_OUT,
      model_name=model.name, learning_rate=LR, batch_size=BATCH, EPOCHS=EPOCHS,
      device=device, early_stop_patience=8)
test_df, metrics = test(model, test_ds, fold_i=FOLD, model_name=model.name,
                        saved_model_path=EXPR_OUT, batch_size=BATCH, device=device)
print(f"\nfold {FOLD} expression Pearson R = {metrics['pearsonr']:.4f}")

### 2c · Evaluate expression  *(12-fold target RNA ≈ 0.86 / CAGE ≈ 0.87)*

As in Part 1, pooled evaluation runs only after all 12 fold prediction files exist in the output
directory for the selected `EXPR_TYPE`.

In [ ]:
from pathlib import Path
import subprocess
import sys

expression_pred_dir = Path(EXPR_OUT)
missing_folds = [
    fold for fold in range(1, 13)
    if not list(expression_pred_dir.glob(f"fold_{fold}_*_predictions.csv"))
]
if missing_folds:
    print(f"{EXPR_TYPE} pooled evaluation skipped; missing folds: {missing_folds}")
    print("Run the 12-fold SLURM array, then rerun this cell.")
else:
    subprocess.run(
        [sys.executable, "evaluate.py", "expression", "--pred_dir", EXPR_OUT],
        check=True,
    )

---
## 3 · Inspect your K562 results

The cells below load the summaries and figures written by `evaluate.py`. They're safe to run
anywhere — if a file isn't present yet (results still on the HPC), they just say so. To pull results
down first:

```bash
rsync_from_remote results/seqencoder/K562_repro        # encoder preds + summary + scatter
rsync_from_remote EPInformer_models/K562_repro_RNA_prm # expression preds + summary + scatter
```

In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

# Each row carries its own target so RNA, CAGE, and encoder protocols stay correctly labeled.
RESULTS = {
    "Encoder - single reverse": ("results/seqencoder/K562_repro", "encoder", 0.734),
    "Encoder - fwd+RC":         ("results/seqencoder/K562_repro_fwdRC", "encoder", 0.740),
    "Expression - RNA":         ("EPInformer_models/K562_repro_RNA_prm", "expression", 0.86),
    # "Expression - CAGE":      ("EPInformer_models/K562_repro_CAGE_prm", "expression", 0.87),
}

for title, (pred_dir, mode, target) in RESULTS.items():
    print("=" * 64)
    print(f"{title}  ->  {pred_dir}")
    summ = os.path.join(pred_dir, f"{mode}_eval_summary.csv")
    if os.path.exists(summ):
        df = pd.read_csv(summ)
        overall = df[df["fold"].astype(str) == "ALL"]
        if len(overall):
            r = float(overall["pearsonr"].iloc[0])
            print(f"  pooled out-of-fold Pearson R = {r:.4f}   (target ~{target})")
        else:
            print(df.to_string(index=False))
    else:
        print("  (no eval summary yet - run evaluate.py, or rsync results from the HPC)")
    png = os.path.join(pred_dir, f"{mode}_scatter.png")
    if os.path.exists(png):
        display(Image(filename=png))
    else:
        print("  (no scatter png yet)")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

panels = [
    ("Encoder",          "results/seqencoder/K562_repro",         "encoder"),
    ("Expression (RNA)", "EPInformer_models/K562_repro_RNA_prm",  "expression"),
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (title, pred_dir, mode) in zip(axes, panels):
    summ = os.path.join(pred_dir, f"{mode}_eval_summary.csv")
    if not os.path.exists(summ):
        ax.set_title(f"{title}\n(no results yet)", fontsize=13)
        ax.axis("off")
        continue
    df = pd.read_csv(summ)
    per_fold = df[df["fold"].astype(str) != "ALL"].copy()
    per_fold["fold"] = per_fold["fold"].astype(int)
    per_fold = per_fold.sort_values("fold")
    ax.bar(per_fold["fold"].astype(str), per_fold["pearsonr"])
    pooled = df[df["fold"].astype(str) == "ALL"]["pearsonr"]
    if len(pooled):
        ax.axhline(float(pooled.iloc[0]), color="crimson", ls="--",
                   label=f"pooled {float(pooled.iloc[0]):.3f}")
        ax.legend(fontsize=11)
    ax.set_title(f"K562 {title} - per-fold Pearson R", fontsize=13)
    ax.set_xlabel("fold", fontsize=12)
    ax.set_ylabel("Pearson R", fontsize=12)
plt.tight_layout()
plt.show()

---
## Troubleshooting & gotchas (K562)

- **Order matters:** Part 2 loads the encoder checkpoint for the same fold from Part 1.
- **One fold is illustrative:** the in-cell result is a fold metric, not the headline pooled result.
  The evaluation cells require all 12 fold files; use the SLURM arrays to produce them efficiently.
- **Run All is safe:** preprocessing cells default to preview-only. Opt in separately to downloads
  and SLURM submission after checking storage, inputs, and the cluster queue.
- **Lustre HDF5:** run the preflight cell before importing HDF5-backed training modules. Bash cells
  also export `HDF5_USE_FILE_LOCKING=FALSE` for their own subprocesses.
- **RNA and CAGE stay separate:** `EXPR_OUT` is derived from `EXPR_TYPE`; do not combine prediction
  files from different targets or model variants.
- **Encoder protocols stay separate:** `K562_repro` is single-reverse (~0.734); write forward+RC
  predictions to `K562_repro_fwdRC` (~0.740).
- **GPU safety:** full training stops when CUDA is unavailable. Explicitly enable non-CUDA mode only
  for a short smoke test, not for reproduction metrics.
- **SLURM queue:** submit to `gpu33` first; fall back to `gpu31` / `gpu2` if needed. Use `cpu1` only
  for preprocessing and other CPU-only jobs.
- **Encoder directory:** do not use the older `results/seqencoder/K562` 3-bin/MSE baseline.
- **Kernel:** select **EPInformer (ep_env)**. A generic `python3` kernel may lack torch or kipoiseq.